In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_33.pickle

In [3]:
%%RecordEvent
%%time
### cell 33 ###

def grab_subset_of_data_45(df, question_of_interest):
    # Select only the columns that contain the question and rename them
    cols = [col for col in df.columns if question_of_interest in col]
    new_names = [col.split('-')[-1].lstrip() for col in cols]
    subset = df[cols].copy()
    subset.columns = new_names
    # Drop rows where all of these columns are NaN
    return subset.dropna(how='all')

# Grab and preprocess the subset
ml_frameworks_df_2022_cell45 = grab_subset_of_data_45(
    responses_df_2022_cell10,
    question_of_interest_cell44
)

# Define the groups of columns to combine
groups = {
    'TensorFlow/Keras': ['TensorFlow ', 'Keras '],
    'PyTorch/PyTorch Lightning/Fast.ai': ['PyTorch ', 'PyTorch Lightning ', 'Fast.ai '],
    'Xgboost/LightGBM/CatBoost': ['Xgboost ', 'LightGBM ', 'CatBoost '],
}
# Flatten the list of all original columns to drop later
drop_cols = [col for cols in groups.values() for col in cols]
# Compute a single DataFrame of boolean masks to avoid repeated .notna() calls
bool_df = ml_frameworks_df_2022_cell45[drop_cols].notna()

# Create each combined column by mapping True→label, False→NaN
for label, cols in groups.items():
    mask = bool_df[cols].any(axis=1)
    ml_frameworks_df_2022_cell45[label] = mask.map({True: label})

# Drop the original detailed columns and reorder
ml_frameworks_df_2022_cell45 = (
    ml_frameworks_df_2022_cell45
    .drop(columns=drop_cols)
    [[ 'Scikit—learn '] + list(groups.keys())]
)

ml_frameworks_df_2022_cell45.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14531 entries, 20598 to 2732
Data columns (total 4 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Scikit—learn                       11403 non-null  object
 1   TensorFlow/Keras                   8850 non-null   object
 2   PyTorch/PyTorch Lightning/Fast.ai  5516 non-null   object
 3   Xgboost/LightGBM/CatBoost          4931 non-null   object
dtypes: object(4)
memory usage: 567.6+ KB
CPU times: user 41.2 ms, sys: 192 µs, total: 41.4 ms
Wall time: 41.4 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_33_try_0.pickle